# Model Evaluation on Held-Out Test Set

Evaluates every trained model against `test_data/VisDrone2019-DET-test-dev` — data none of the models have seen before.

**Steps:**
1. Convert VisDrone annotations → YOLO label format (cached — skipped if already done)
2. Write a `visdrone_test.yaml` pointing at the test split
3. Run `model.val()` for every model
4. Rank all models by mAP@50-95 and display a summary table + bar chart

In [6]:
from pathlib import Path
import shutil
import yaml

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from PIL import Image
from ultralytics import YOLO

In [7]:
# ── Absolute paths (notebook lives in analyze/, everything else is one level up)
ROOT        = Path("../").resolve()
TEST_RAW    = ROOT / "test_data" / "VisDrone2019-DET-test-dev"
MODELS_DIR  = ROOT / "models"
YAML_PATH   = ROOT / "test_data" / "visdrone_test.yaml"

# ── Model registry: label → (weights path, imgsz used during training)
# Adjust imgsz here if any model was trained with a non-default size
MODELS = {
    "baseline":          (MODELS_DIR / "baseline.pt",                     640),
    "adamw":             (MODELS_DIR / "experiment" / "adamw.pt",          640),
    "adamw_imghigh":     (MODELS_DIR / "experiment" / "adamw_imghigh.pt", 1280),
    "cosine_lr_imghigh": (MODELS_DIR / "experiment" / "cosine_lr_imghigh.pt", 1280),
    "data_aug":          (MODELS_DIR / "experiment" / "data_aug.pt",        640),
    "imgsize_high":      (MODELS_DIR / "experiment" / "imgsize_high.pt",   1280),
    "improvment_cycle":  (MODELS_DIR / "improvment_cycle.pt",               640),
    "yolov8":            (MODELS_DIR / "verisoned" / "yolov8.pt",           640),
    "yolov9":            (MODELS_DIR / "verisoned" / "yolov9.pt",           640),
    "yolov10":           (MODELS_DIR / "verisoned" / "yolov10.pt",          640),
}

# ── VisDrone class mapping (cat 0 = ignored region, cat 11 = others → skipped)
VISDRONE_CLASSES = {
    1: 0,   # pedestrian
    2: 1,   # people
    3: 2,   # bicycle
    4: 3,   # car
    5: 4,   # van
    6: 5,   # truck
    7: 6,   # tricycle
    8: 7,   # awning-tricycle
    9: 8,   # bus
    10: 9,  # motor
}

CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
]

print(f"ROOT       : {ROOT}")
print(f"TEST_RAW   : {TEST_RAW}")
print(f"Models     : {len(MODELS)}")

ROOT       : /Users/samsikora/Desktop/Desktop/UBC/4.2/CMPE 401/Assignments/Yolo/yolo-object-detection-model-analysis
TEST_RAW   : /Users/samsikora/Desktop/Desktop/UBC/4.2/CMPE 401/Assignments/Yolo/yolo-object-detection-model-analysis/test_data/VisDrone2019-DET-test-dev
Models     : 10


## Step 1 — Convert annotations (cached)

In [8]:
DST_DIR    = ROOT / "test_data" / "data"   # mirrors the Colab data/ layout

def convert_split(src_dir, split_name):
    src_dir    = Path(src_dir)
    images_dir = src_dir / "images"
    annots_dir = src_dir / "annotations"

    out_images = DST_DIR / "images" / split_name
    out_labels = DST_DIR / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    all_images = sorted(images_dir.glob("*.jpg"))
    total      = len(all_images)

    print(f"\n{'='*60}")
    print(f"  Converting split : {split_name.upper()}")
    print(f"  Source           : {src_dir}")
    print(f"  Images found     : {total}")

    # ── Cache check: skip images already converted ─────────────────────────────
    pending      = []
    already_done = 0
    for img_path in all_images:
        out_img   = out_images / img_path.name
        out_label = out_labels / (img_path.stem + ".txt")
        if out_img.exists() and out_label.exists():
            already_done += 1
        else:
            pending.append(img_path)

    if already_done > 0:
        print(f"  Cache hit        : {already_done} / {total} already converted — skipping")
    if not pending:
        print(f"  ✅ Split '{split_name}' fully cached — nothing to do!")
        print(f"{'='*60}")
        return
    print(f"  To process       : {len(pending)} images")
    print(f"{'='*60}")

    # ── Process only pending images ────────────────────────────────────────────
    skipped_imgs  = 0
    total_boxes   = 0
    skipped_boxes = 0

    for idx, img_path in enumerate(pending, 1):
        annot_path = annots_dir / (img_path.stem + ".txt")

        # Progress every 500 images
        if idx % 500 == 0 or idx == len(pending):
            print(f"  [{idx:>5}/{len(pending)}] Processing {img_path.name} ...")

        if not annot_path.exists():
            print(f"  [WARN] No annotation for {img_path.name} — skipping")
            skipped_imgs += 1
            continue

        img          = Image.open(img_path)
        img_w, img_h = img.size

        yolo_lines = []
        with open(annot_path) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    skipped_boxes += 1
                    continue

                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                score       = int(parts[4])
                cat         = int(parts[5])

                # Skip ignored regions, 'others', and score=0 (ignored annotations)
                if cat not in VISDRONE_CLASSES or score == 0:
                    skipped_boxes += 1
                    continue

                yolo_cat = VISDRONE_CLASSES[cat]
                cx = max(0.0, min(1.0, (x + w / 2) / img_w))
                cy = max(0.0, min(1.0, (y + h / 2) / img_h))
                nw = max(0.0, min(1.0, w / img_w))
                nh = max(0.0, min(1.0, h / img_h))

                if nw > 0 and nh > 0:
                    yolo_lines.append(f"{VISDRONE_CLASSES[cat]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                    total_boxes += 1
                else:
                    skipped_boxes += 1

        shutil.copy(img_path, out_images / img_path.name)
        with open(out_labels / (img_path.stem + ".txt"), "w") as f:
            f.write("\n".join(yolo_lines))

    print(f"\n  ✅ Done — {split_name.upper()}")
    print(f"     Images converted : {len(pending) - skipped_imgs} / {len(pending)} pending")
    print(f"     Bounding boxes   : {total_boxes} kept  |  {skipped_boxes} skipped (ignored/invalid)")
    print(f"     Output images    : {out_images}")
    print(f"     Output labels    : {out_labels}")


convert_split(TEST_RAW, "test")



  Converting split : TEST
  Source           : /Users/samsikora/Desktop/Desktop/UBC/4.2/CMPE 401/Assignments/Yolo/yolo-object-detection-model-analysis/test_data/VisDrone2019-DET-test-dev
  Images found     : 1610
  Cache hit        : 1610 / 1610 already converted — skipping
  ✅ Split 'test' fully cached — nothing to do!


## Step 2 — Write dataset YAML

In [9]:
yaml_content = f"""\
path: {DST_DIR}
train: images/test
val:   images/test
test:  images/test

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = DST_DIR / "visdrone.yaml"
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"  YAML saved → {yaml_path}\n")
print(yaml_content)


  YAML saved → /Users/samsikora/Desktop/Desktop/UBC/4.2/CMPE 401/Assignments/Yolo/yolo-object-detection-model-analysis/test_data/data/visdrone.yaml

path: /Users/samsikora/Desktop/Desktop/UBC/4.2/CMPE 401/Assignments/Yolo/yolo-object-detection-model-analysis/test_data/data
train: images/test
val:   images/test
test:  images/test

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor



## Step 3 — Evaluate all models

In [10]:
records = []

for name, (weights, imgsz) in MODELS.items():
    if not weights.exists():
        print(f"[SKIP] {name}: weights not found at {weights}")
        continue

    print(f"\n{'='*55}")
    print(f"  Evaluating : {name}  (imgsz={imgsz})")
    print(f"{'='*55}")

    model   = YOLO(str(weights))
    metrics = model.val(
        data   = str(yaml_path),
        split  = "test",
        imgsz  = imgsz,
        verbose= False,
    )

    spd = metrics.speed   # ms/image: {preprocess, inference, postprocess}
    records.append({
        "model":              name,
        "mAP@50-95":          round(metrics.box.map,   4),
        "mAP@50":             round(metrics.box.map50,  4),
        "precision":          round(metrics.box.mp,     4),
        "recall":             round(metrics.box.mr,     4),
        "preprocess (ms)": round(spd["preprocess"],  3),
        "inference (ms)":  round(spd["inference"],   3),
        "postprocess (ms)": round(spd["postprocess"], 3),
        "total_ms/img":    round(sum(spd.values()),   3),
    })

    print(f"  mAP@50-95 : {metrics.box.map:.4f}")
    print(f"  mAP@50    : {metrics.box.map50:.4f}")
    print(f"  Precision : {metrics.box.mp:.4f}")
    print(f"  Recall    : {metrics.box.mr:.4f}")
    print(f"  Preprocess: {spd['preprocess']:.1f} ms/img")
    print(f"  Inference : {spd['inference']:.1f} ms/img")
    print(f"  Postproc  : {spd['postprocess']:.1f} ms/img")
    print(f"  Total     : {sum(spd.values()):.1f} ms/img")

print("\n\nAll evaluations complete.")


## Step 4 — Rank and visualise

In [ ]:
df = (
    pd.DataFrame(records)
    .sort_values("mAP@50-95", ascending=False)
    .reset_index(drop=True)
)
df.index += 1          # rank starts at 1
df.index.name = "rank"

print("\n" + "="*60)
print("       MODEL RANKINGS — VisDrone Test Set")
print("="*60)
print(df.to_string())
print("="*60)

# Save to CSV next to notebook
out_csv = Path("model_rankings.csv")
df.to_csv(out_csv)
print(f"\nSaved → {out_csv.resolve()}")

df

In [ ]:
metrics_to_plot = ["mAP@50-95", "mAP@50", "precision", "recall"]
n_models  = len(df)
colors    = [cm.tab10(i / 10.0) for i in range(n_models)]
x         = np.arange(n_models)
bar_width = 0.18

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Model Rankings — VisDrone Held-Out Test Set", fontsize=14, fontweight="bold")

# ── Top-left: grouped bar — all accuracy metrics ──────────────────────────────
ax = axes[0, 0]
for m_idx, metric in enumerate(metrics_to_plot):
    offsets = (m_idx - (len(metrics_to_plot) - 1) / 2) * bar_width
    ax.bar(x + offsets, df[metric], width=bar_width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(df["model"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Accuracy Metrics by Model (ranked by mAP@50-95)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1)

# ── Top-right: horizontal bar — mAP@50-95 ranking ────────────────────────────
ax2 = axes[0, 1]
y_pos = np.arange(n_models)[::-1]
ax2.barh(y_pos, df["mAP@50-95"], color=colors, edgecolor="white", height=0.6)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(df["model"], fontsize=9)
ax2.set_xlabel("mAP@50-95")
ax2.set_title("mAP@50-95 Rankings")
ax2.grid(axis="x", alpha=0.3)
ax2.set_xlim(0, max(df["mAP@50-95"]) * 1.15)
for val, pos in zip(df["mAP@50-95"], y_pos):
    ax2.text(val + 0.001, pos, f"{val:.4f}", va="center", fontsize=8)

# ── Bottom-left: stacked bar — inference time breakdown ───────────────────────
ax3 = axes[1, 0]
speed_cols  = ["preprocess (ms)", "inference (ms)", "postprocess (ms)"]
speed_colors = ["#4e9af1", "#f4a742", "#6dcf7f"]
bottoms = np.zeros(n_models)
for col, color in zip(speed_cols, speed_colors):
    vals = df[col].values
    ax3.bar(x, vals, bottom=bottoms, label=col, color=color, edgecolor="white")
    bottoms += vals
ax3.set_xticks(x)
ax3.set_xticklabels(df["model"], rotation=30, ha="right", fontsize=9)
ax3.set_ylabel("ms / image")
ax3.set_title("Inference Time Breakdown (ms/image)")
ax3.legend(fontsize=9)
ax3.grid(axis="y", alpha=0.3)

# ── Bottom-right: scatter — accuracy vs speed trade-off ───────────────────────
ax4 = axes[1, 1]
for i, row in df.iterrows():
    ax4.scatter(row["total_ms/img"], row["mAP@50-95"],
                color=colors[i - 1], s=100, zorder=3)
    ax4.annotate(row["model"], (row["total_ms/img"], row["mAP@50-95"]),
                 textcoords="offset points", xytext=(6, 3), fontsize=8)
ax4.set_xlabel("Total inference time (ms/image)")
ax4.set_ylabel("mAP@50-95")
ax4.set_title("Accuracy vs Speed Trade-off")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
out_plot = Path("model_rankings.png")
plt.savefig(out_plot, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {out_plot.resolve()}")
